# SHAP for Text and Image Data

SHAP values can be computed for non-tabular data by defining appropriate maskers and partition-based explainers. For text, SHAP treats words/tokens as features and measures each token's contribution to the prediction. For images, SHAP uses superpixels as features to identify which regions of an image are most important for the model's decision.

In [ ]:
import shap
import transformers
import numpy as np
import matplotlib.pyplot as plt

## SHAP for Text Classification

In [ ]:
# Load a sentiment analysis pipeline from transformers
model = transformers.pipeline('sentiment-analysis', return_all_scores=True)

# Create a SHAP explainer for the model
explainer = shap.Explainer(model)

# Texts to explain
texts = [
    "This movie was excellent and I loved it!",
    "The food was terrible and the service was slow.",
    "Amazing product, highly recommended to everyone."
]

# Compute SHAP values
shap_values = explainer(texts)

In [ ]:
# Visualize SHAP values for text
shap.plots.text(shap_values)

## SHAP for Image Classification

For image data, SHAP uses partition explainers with image maskers that treat superpixels or regions as features. This approach measures how important different parts of an image are for the model's prediction, allowing us to identify which regions the model focuses on.

In [ ]:
import json
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions

# Load pretrained ResNet50 model
model = ResNet50(weights='imagenet')

# Define a function that takes masked images and returns predictions
def f(x):
    tmp = preprocess_input(x.copy())
    return model(tmp)

# Use SHAP's sample image dataset
X, y = shap.datasets.imagenet50()

# Create masker and explainer
masker = shap.maskers.Image("inpaint_telea", X[0].shape)
explainer = shap.Explainer(f, masker, output_names=decode_predictions(model.predict(preprocess_input(X[:1].copy())), top=3)[0])

# Explain the first image
shap_values = explainer(X[:1], max_evals=500, batch_size=50)

In [ ]:
# Visualize SHAP values for images
shap.image_plot(shap_values)

## Interpretation

In text visualizations, red-colored tokens indicate words that increase the predicted class probability (positive contribution), while blue tokens decrease it (negative contribution). Similarly, for images, red regions are superpixels that push the prediction toward the classified object, while blue regions push against it, allowing us to understand which visual features the model relies on for its decision.